# 4.5.3 Efficient Attention Variants

Full O(n²) vs Linear O(n) vs Sliding Window O(n·w) — complexity and scaling.

In [ ]:
import numpy as np, time

def softmax(x,ax=-1):
    x=x-x.max(axis=ax,keepdims=True); e=np.exp(x); return e/e.sum(axis=ax,keepdims=True)

def full_attn(Q,K,V): return softmax(Q@K.T/np.sqrt(Q.shape[-1]))@V

def linear_attn(Q,K,V):
    Q=np.maximum(Q+1,1e-6); K=np.maximum(K+1,1e-6)
    KV=K.T@V; Z=Q@K.sum(0,keepdims=True).T
    return Q@KV/(Z+1e-8)

rng=np.random.default_rng(42)
for seq in [16,32,64,128]:
    Q=rng.standard_normal((seq,8)); K=rng.standard_normal((seq,8)); V=rng.standard_normal((seq,8))
    t0=time.perf_counter(); full_attn(Q,K,V); tf=time.perf_counter()-t0
    t0=time.perf_counter(); linear_attn(Q,K,V); tl=time.perf_counter()-t0
    print(f'seq={seq:4d}: full={tf*1e3:.2f}ms  linear={tl*1e3:.2f}ms')

In [ ]:
# Complexity classes
vars={'Full O(n²)':lambda n:n**2,'Linear O(n)':lambda n:n,'Window O(n·4)':lambda n:4*n}
import matplotlib.pyplot as plt
ns=range(8,256,8)
for name,fn in vars.items(): plt.plot(list(ns),[fn(n) for n in ns],label=name)
plt.xlabel('Seq len'); plt.title('Ops Complexity'); plt.legend(); plt.show()